# VaR forecast coverage checks

Do simple daily loss forecasts produce the expected number of breaches?

Authored 2026-10-03. AI-assisted implementation. Read the study status and data boundaries.

Create lagged 250-session historical and exponentially weighted Gaussian risk forecasts at 95% and 99%. Count later exceedances, show Wilson coverage intervals, Kupiec unconditional-coverage likelihood ratios and descriptive adjacent-breach counts.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'investment_lab').exists():
    root = root.parent
if not (root / 'investment_lab').exists():
    raise RuntimeError('Open this notebook from the project root or notebooks folder')
sys.path.insert(0, str(root))
from investment_lab.runner import run_project, summary


In [2]:
result = run_project('varchecks')
print(summary('varchecks', result))
print('\nLimitation:', result['limitation'])

Rolling historical 99% VaR recorded 9 breaches in 484 evaluation sessions (expected 4.8). Coverage and clustering must be examined together.

Limitation: Neither model certifies tail protection. Small breach counts make asymptotic p-values fragile; no conditional-independence test or regulatory approval is claimed. Model selection is retrospective, observations are dependent and realised P&L is constructed with later weights. Do not use a passed coverage check as proof of a sound expected-shortfall model.


In [3]:
# Inspect the full sensitivity tables, not just the headline.
for name, rows in result['tables'].items():
    print(name, ':', len(rows), 'rows')
    for row in rows[:3]:
        print(row)
print('Input assumptions:', *result['inputs'], sep='\n- ')

coverage : 4 rows
{'model': 'historical', 'tail_probability': 0.05, 'observations': 484, 'exceptions': 27, 'expected_exceptions': 24.200000000000003, 'exception_rate': 0.05578512396694215, 'wilson_lower95': 0.03861869294534753, 'wilson_upper95': 0.0799476613794989, 'kupiec_lr': 0.32923418197412957, 'asymptotic_p_value': 0.566110377792638, 'adjacent_breach_pairs': 1}
{'model': 'historical', 'tail_probability': 0.01, 'observations': 484, 'exceptions': 9, 'expected_exceptions': 4.84, 'exception_rate': 0.01859504132231405, 'wilson_lower95': 0.009813036100301882, 'wilson_upper95': 0.03495887347034564, 'kupiec_lr': 2.8817989318736394, 'asymptotic_p_value': 0.08958588781058069, 'adjacent_breach_pairs': 1}
{'model': 'ewma', 'tail_probability': 0.05, 'observations': 484, 'exceptions': 24, 'expected_exceptions': 24.200000000000003, 'exception_rate': 0.049586776859504134, 'wilson_lower95': 0.03354543106371009, 'wilson_upper95': 0.07272184913543651, 'kupiec_lr': 0.0017444466293881078, 'asymptotic_

![Computed chart](../reports/varchecks.png)

## Investment implication

Check both frequency and clustering before relying on a risk number. A model that passes a small-sample coverage check can still miss crisis losses.

## What could invalidate this interpretation?

The fixed current-weight proxy is hindsight-based. Gaussian tails, sparse 99% exceptions and asymptotic p-values limit inference; clustering is not formally tested here.

## Investigate next

Maintain a prospective ledger of forecasts and realised hypothetical P&L; add independent clustering and tail-severity tests.

Full numeric outputs and CSVs: `reports/varchecks.json` and `reports/varchecks_*.csv`. References: `reports/varchecks.md`.